# Calculating semantic tag statistics for verb-case pairings

I am trying to answer the research question: How much can a verb and its dependent's case determine the dependent's semantic type?

This notebook gathers statistics in order to answer that question. It is used to calculate various statistical information about semantic tags for verb and case pairs and put the results into database tables.

For that purpose we go through the following steps:
1. Separate the case tag from the larger feats value.
2. Count for each verb+case pair how many dependents were annotated as an user specified tag, other tags, semantically annotated at all or recieved no semantic tag
3. Calculate percentages for how many of the verb+case pairs had words that are an user specified tag, other tags, annotated and not_annotated based on the counts from step 2.
4. Calculate proportion of location/not_location and annotated/not_annotated with binary logarithms. These will later be used to make graph showing if a verb's dependents in a specific case are more likely some user specified tags or other tags and how many of the words are annotated at all
5. Find how many unique lemmas each verb+case pair has, calculating its binary algorithm. Used later as a hoverplot's x-axis to show how much the tag proportions can be trusted.

In [1]:
import sqlite3
import pandas as pd
import numpy as np

In [2]:
# database file path
filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

# connecting with database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

## 1. Separate cases
Separating the case tag from the larger feats value and adding it to the database table

#### Add new column for case

In [3]:
conn = sqlite3.connect(filename)
cursor = conn.cursor()
cursor.execute("ALTER TABLE spatial_obl ADD COLUMN morph_case TEXT")

#### Separate case from feats column

In [4]:
cases = ['adit', 'ill', 'in', 'el', 'all', 'ad', 'abl']

# Fetch feats column
cursor.execute("SELECT id, feats FROM spatial_obl")
rows = cursor.fetchall()

# find case and separate into separate column
updates = []
for rowid, feats in rows:
    if feats:
        for case in cases:
            if case in feats.split(","):
                case_value = case
        updates.append((case_value, rowid))

#### Add case info to case column

In [5]:
# Step 1: Create a temporary table
cursor.execute("CREATE TEMP TABLE temp_case (id INT PRIMARY KEY, morph_case TEXT)")

# Step 2: Insert all values into the temp table
cursor.executemany("INSERT INTO temp_case (morph_case, id) VALUES (?, ?)", updates)

# Step 3: Perform a fast join-based update
cursor.execute("""
    UPDATE spatial_obl
    SET morph_case = (SELECT morph_case FROM temp_case WHERE temp_case.id = spatial_obl.id)
""")

# Commit changes and close connection
conn.commit()
conn.close()

## 2. Count semantic tags
Create a table with pure counts where each verb+case tag has how many of that verb's dependents in that case have user defined semantic tags, other tags, are annotated, aren't annotated and how many times the verb took a dependent in that case

The user has to insert the semantic tags they want counts of while calling the function *count_table*

In [5]:
#create a table that counts tags for verb+case pairs
#tag1 = first tag to count
#tag2 = tag you want to count together with the second tag
def count_table(tag1, tag2 = ''):
    #Connect to database
    conn = sqlite3.connect(filename)
    cursor = conn.cursor()

    #define tag name in database table
    if tag2 == '':
        tag = tag1
    else:
        tag = tag1 + '_' + tag2
    
    #delete table if it exists
    cursor.execute("DROP TABLE IF EXISTS verb_case_counts_"+tag)

    # Step 1: Create the new counts table 
    cursor.execute("""
            CREATE TABLE IF NOT EXISTS verb_case_counts_"""+tag+""" (
            verb TEXT,
            verb_compound TEXT,
            morph_case TEXT, 
            my_tag INT,
            other_tags INT,
            annotated INT,
            not_annotated INT,
            verb_case_count INT
        )
    """)

    # Step 2: Aggregate counts
    cursor.execute(
    f"""INSERT INTO verb_case_counts_{tag}
        (verb, verb_compound, morph_case, my_tag, other_tags, annotated, not_annotated, verb_case_count)
        SELECT 
            verb, 
            verb_compound,
            morph_case, 
            COUNT(CASE WHEN ekilex_tag = ? OR ekilex_tag = ? THEN 1 END) AS my_tag,
            COUNT(CASE WHEN ekilex_tag IS NOT NULL AND ekilex_tag != ? AND ekilex_tag != ? THEN 1 END) AS other_tags,
            COUNT(CASE WHEN ekilex_tag IS NOT NULL THEN 1 END) AS annotated,
            COUNT(CASE WHEN ekilex_tag IS NULL THEN 1 END) AS not_annotated,
            COUNT(*) AS verb_case_count
        FROM spatial_obl
        GROUP BY verb, verb_compound, morph_case
    """, (tag1, tag2, tag1, tag2)
    )

    # Commit and close
    conn.commit()
    conn.close()

In [6]:
#create count table for location vs other tags
count_table('location')
#create count table for location+time vs other tags
count_table('location', 'time')

## 3. Calculate percentages
Uses the counts from the previous table to calculate percentages of each class for every verb+case pair
* my_tag_pr = what percentage of annotated words had an user specified tag. User can specify multiple tags
* other_tags_pr: what percentage of annotated words didn't have those tags
* annotated_pr: what percentage of words were annotated
* not_annotated_pr: what percentage of words were not annotated

Users have to define database table name

Results are put into a new dataframe with a verb, verb compund, case and the percentages specified above

In [3]:
def semtype_percentages(table_name):
    #connect to database
    conn = sqlite3.connect(filename)
    cursor = conn.cursor()
    
    #read counts into dataframe
    df = pd.read_sql("SELECT * FROM " + table_name, conn)

    #only take rows that have more than 4 examples
    df = df.loc[df['verb_case_count'] > 4]

    #remove verb+case pairs that have no annotated dependents
    df = df.loc[df['annotated'] != 0]

    # Calculate percentages
    df["my_tag_pr"] = df["my_tag"] / df["annotated"]
    df["other_tags_pr"] = df["other_tags"] / df["annotated"]
    df["annotated_pr"] = df["annotated"] / df["verb_case_count"]
    df["not_annotated_pr"] = df["not_annotated"] / df["verb_case_count"]

    #create a new dataframe with only percentages
    df_pr = df[['verb', 'verb_compound', 'morph_case', "my_tag_pr", 'other_tags_pr', 'annotated_pr', 'not_annotated_pr', 'verb_case_count']].copy()

    # Commit and close
    conn.commit()
    conn.close()

    return df_pr

In [4]:
#percentage dataframe for location
df_pr_loc = semtype_percentages('verb_case_counts_location')
#percentage dataframe for location+time
df_pr_loctime = semtype_percentages('verb_case_counts_location_time')
#example
df_pr_loctime.loc[df_pr_loctime['verb'] == 'käima']

,verb,verb_compound,morph_case,my_tag_pr,other_tags_pr,annotated_pr,not_annotated_pr,verb_case_count
22906,käima,,abl,0.569492,0.430508,0.362854,0.637146,813
22907,käima,,ad,0.626140,0.373860,0.335141,0.664859,41869
22908,käima,,adit,0.757085,0.242915,0.138764,0.861236,1780
22909,käima,,all,0.303178,0.696822,0.169639,0.830361,4822
22910,käima,,el,0.603097,0.396903,0.282394,0.717606,8003
...,...,...,...,...,...,...,...,...
23221,käima,üle,in,0.680851,0.319149,0.412281,0.587719,114
23225,käima,üles,el,1.000000,0.000000,0.800000,0.200000,5
23226,käima,üles,in,1.000000,0.000000,0.200000,0.800000,5
23228,käima,ümber,ad,1.000000,0.000000,0.035714,0.964286,28


## 4. Calculate proportions
To better illustrate whether a verb+case pair prefers the user specified tag (ie location) or all the other tags (ie time+state+event), we calculate pointwise mutual information by dividing an user specified tag(s) percentage by every other_tags percentage and taking a binary logarithm of it. 

This section:
* Calculates PMI for tag vs other_tags and annotated vs not_annotated per verb + case pair
* Adds logarithms to dataframe and
* Transforms the dataframe into a database table

In [5]:
def semtype_log(df_pr, table_name):

    #replace zeros with 0,001 to avoid taking log from zero
    #not replacing zero with a VERY small number like 1e-10 to avoid graph stretching out
    df_filt = df_pr.replace(0.0, 0.001)

    #calculate binary logarithm for specific tag(s) vs other tags
    df_filt["log2_tag"] = np.log2((df_filt["my_tag_pr"]) / (df_filt["other_tags_pr"]))

    #calculate binary logarithm for annotated/not_annotated
    df_filt["log2_annotation"] = np.log2((df_filt["annotated_pr"]) / (df_filt["not_annotated_pr"]))

    #create a new dataframe with only the proportions
    df_log2 = df_filt[['verb', 'verb_compound', 'morph_case', 'log2_tag', 'log2_annotation', 'verb_case_count']].copy()
    
    #connect to database
    conn = sqlite3.connect(filename)
    cursor = conn.cursor()

    #drop table if it exists
    cursor.execute("DROP TABLE IF EXISTS " + table_name)

    # Step 1: Create the new results table 
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS """ +  table_name  + """ (
            verb TEXT,
            verb_compound TEXT,
            morph_case TEXT,
            log2_tag REAL,
            log2_annotation REAL,
            verb_case_count INT
        )
    """)

    # Step 2: Insert percentages from the dataframe into the database table
    df_log2.to_sql(table_name, conn, if_exists="replace", index=False)

    # Commit and close
    conn.commit()
    conn.close()

In [6]:
#create log table for locations
semtype_log(df_pr_loc, 'verb_case_log_location')
#create log table for locations + time
semtype_log(df_pr_loctime, 'verb_case_log_location_time')

## 5. Find how many different words each verb+case pair has

This is to show how trustworthy the tag proportions are for a verb+case pair. The more differents words a verb's dependents in said case are, the more trustworthy the results are, because the sample was larger

#### Find unique lemma counts for verb+case pairs, calculate binary logarithm for them

In [7]:
#Connect to database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

# Aggregate counts
query = """
    SELECT 
        verb, 
        verb_compound,
        morph_case, 
        COUNT(DISTINCT lemma) AS unique_lemmas
    FROM spatial_obl
    WHERE ekilex_tag is not null
    GROUP BY verb, verb_compound, morph_case
"""

# Make into dataframe to calculate log2
df_unique_lemmas = pd.read_sql(query, conn)

#calculate binary logarithm for unique lemmas
df_unique_lemmas["log2_unique_lemmas"] = np.log2(df_unique_lemmas["unique_lemmas"]) 

#### Create a new database table for the unique lemma counts and logarithms

In [8]:
#delete table if it exists
cursor.execute("DROP TABLE IF EXISTS verb_case_unique_lemmas")

# Step 1: Create the new counts table 
cursor.execute("""
    CREATE TABLE IF NOT EXISTS verb_case_unique_lemmas (
        verb TEXT,
        verb_compound TEXT,
        morph_case TEXT,
        unique_lemmas INT,
        log2_unique_lemmas REAL
    )
""")

df_unique_lemmas.to_sql("verb_case_unique_lemmas", conn, if_exists="replace", index=False)

# Commit and close
conn.commit()
conn.close()

#### Add unique lemma columns to log statistics tables to ease hoverplot creation

In [9]:
def new_columns(table_name):
    #Connect to database
    conn = sqlite3.connect(filename)
    cursor = conn.cursor()

    cursor.execute("""
        ALTER TABLE """ + table_name + """
        ADD COLUMN unique_lemmas INTEGER;
    """)

    cursor.execute("""
        ALTER TABLE """ + table_name + """
        ADD COLUMN log2_unique_lemmas REAL;
    """)

In [10]:
new_columns('verb_case_log_location')
new_columns('verb_case_log_location_time')

#### Add unique lemma count data to log statistics table

In [11]:
def unique_lemma(table_name):
    #Connect to database
    conn = sqlite3.connect(filename)
    cursor = conn.cursor()

    cursor.execute(
        f"""
        UPDATE {table_name}
        SET unique_lemmas = (
            SELECT unique_lemmas
            FROM verb_case_unique_lemmas
            WHERE 
                verb_case_unique_lemmas.verb = {table_name}.verb
                AND verb_case_unique_lemmas.verb_compound = {table_name}.verb_compound
                AND verb_case_unique_lemmas.morph_case = {table_name}.morph_case
    );
    """)

    # Commit and close
    conn.commit()
    conn.close()

In [12]:
unique_lemma('verb_case_log_location')
unique_lemma('verb_case_log_location_time')

#### Add binary logarithm data of unique lemmas to log statistics table

In [13]:
def unique_lemma_log(table_name):
    #Connect to database
    conn = sqlite3.connect(filename)
    cursor = conn.cursor()

    cursor.execute(
        f"""
        UPDATE {table_name}
        SET log2_unique_lemmas = (
            SELECT log2_unique_lemmas
            FROM verb_case_unique_lemmas
            WHERE 
                verb_case_unique_lemmas.verb = {table_name}.verb
                AND verb_case_unique_lemmas.verb_compound = {table_name}.verb_compound
                AND verb_case_unique_lemmas.morph_case = {table_name}.morph_case
    );
    """)

    # Commit and close
    conn.commit()
    conn.close()

In [14]:
unique_lemma_log('verb_case_log_location')
unique_lemma_log('verb_case_log_location_time')